In [1]:
!pip install pandas pyarrow

### Read the README and generation.json

In [5]:
from pathlib import Path
import json

ROOT = Path.cwd().parent                          # veltrix-segmentation/
base = ROOT / "data" / "raw" / "veltrix_medium"

print(base.exists())                              # should print True
print((base / "README.md").read_text(encoding="utf-8"))

with open(base / "generation_log.json") as f:
    print(json.dumps(json.load(f), indent=2)[:2000])

True
# Veltrix synthetic dataset

Synthetic data for **Veltrix Customer and Buyer Segmentation: A Case Study**. Veltrix is a made-up UK estate agency and new-homes developer, and none of the data describes real people. Customers have pseudonymous IDs plus SHA-256 hashes of their email and phone (no raw contact details). Money is in GBP. The data covers 1 January 2023 to 30 September 2026.

## What's in the package

| File | Contents |
|---|---|
| `veltrix_medium_parquet.zip` | **Main project dataset** (Medium scale): 100,000 base customers, about 19.1M rows, Parquet (zstd) |
| `veltrix_small_csv.zip` | Small scale (10,000 customers, about 2M rows) as CSV, for testing the pipeline, Excel and Power BI |
| `veltrix_generate.py` | The generator. Use `--seed 42` (the default) to rebuild exactly the same data |
| `validate.py` | Checks keys, consistency and realism with DuckDB |
| `recoverability_check.py` | Clustering baseline scored against the answer key. **Run this only after Phase 4** |

### Make an inventory of all the tables

In [6]:
import pyarrow.parquet as pq
import pyarrow.dataset as ds

rows = []
for p in sorted(base.glob("*.parquet")):
    pf = pq.ParquetFile(p)
    rows.append({
        "table": p.stem,
        "rows": pf.metadata.num_rows,
        "cols": pf.metadata.num_columns,
        "size_mb": round(p.stat().st_size / 1e6, 2),
    })

# folders (partitioned datasets)
for d in sorted(x for x in base.iterdir() if x.is_dir() and not x.name.startswith("_")):
    dataset = ds.dataset(d, format="parquet")
    rows.append({
        "table": d.name,
        "rows": dataset.count_rows(),
        "cols": len(dataset.schema.names),
        "size_mb": round(sum(f.stat().st_size for f in d.rglob("*.parquet")) / 1e6, 2),
    })

import pandas as pd
pd.DataFrame(rows)

,table,rows,cols,size_mb
0,agents,300,6,0.01
1,campaign_responses,808953,5,6.11
2,campaigns,270,10,0.01
3,customers,102000,19,7.63
4,interactions,532789,8,5.02
5,leads,95804,11,1.51
6,locations,36,9,0.01
7,payments,106199,10,0.77
8,properties,16667,19,0.44
9,site_visits,233593,6,1.44


### Load everything into a dictionary

In [7]:
tables = {p.stem: pd.read_parquet(p) for p in base.glob("*.parquet")}

# partitioned folders
for name in ["property_views", "web_sessions"]:
    tables[name] = pd.read_parquet(base / name)

list(tables.keys())

['agents',
 'campaigns',
 'campaign_responses',
 'customers',
 'interactions',
 'leads',
 'locations',
 'payments',
 'properties',
 'site_visits',
 'transactions',
 'property_views',
 'web_sessions']

### Profile each table

In [8]:
for name, df in tables.items():
    print(f"\n=== {name} === {df.shape}")
    print(df.dtypes.to_string())
    print(df.head(3).to_string())


=== agents === (300, 6)
agent_id                     str
team                         str
branch                       str
seniority                    str
hire_date                 object
propertymark_qualified      bool
     agent_id          team            branch seniority   hire_date  propertymark_qualified
0  VLX-A00001  London Sales  London (Croydon)    Senior  2005-02-17                    True
1  VLX-A00002  London Sales  London (Croydon)    Junior  2020-11-30                   False
2  VLX-A00003  London Sales  London (Croydon)    Senior  2006-07-05                    True

=== campaigns === (270, 10)
campaign_id             str
campaign_name           str
channel                 str
theme                   str
objective               str
planned_audience        str
target_segment       string
start_date           object
end_date             object
spend_gbp           float64
        campaign_id                              campaign_name     channel               theme      

### Check data quality per table

In [9]:
def quick_profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
    })

quick_profile(tables["leads"])

,dtype,nulls,null_%,unique
lead_id,str,0,0.0,95804
customer_id,str,0,0.0,61030
agent_id,str,0,0.0,300
lead_source,str,0,0.0,28
campaign_id,string,64378,67.2,270
intent,str,0,0.0,2
lead_stage,str,0,0.0,7
lost_reason,str,21045,22.0,11
created_date,datetime64[ms],0,0.0,95748
closed_date,object,4380,4.6,1364


#### Query Data with DuckDB

In [10]:
import duckdb

con = duckdb.connect()

con.sql("""
    SELECT *
    FROM 'raw/veltrix_medium/web_sessions/**/*.parquet'
    LIMIT 5
""").df()

IOException: IO Error: No files found that match the pattern "raw/veltrix_medium/web_sessions/**/*.parquet"

In [ ]:
## Count rows per table
for name in ["leads", "customers", "transactions", "payments"]:
    n = con.sql(f"SELECT COUNT(*) FROM 'raw/veltrix_medium/{name}.parquet'").fetchone()[0]
    print(name, n)

IOException: IO Error: No files found that match the pattern "raw/veltrix_medium/leads.parquet"

### Find how tables connect

In [ ]:
from collections import defaultdict

id_map = defaultdict(list)
for name, df in tables.items():
    for col in df.columns:
        if col.endswith("_id") or col == "id":
            id_map[col].append(name)

for col, tbls in id_map.items():
    print(f"{col:20} -> {tbls}")